<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/bokeh_laboratorio_de_codigos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎨 Bokeh: gráficos interactivos en Python
**Laboratorio de Códigos** · Todo lo bueno de Bokeh en un solo notebook.

Cada celda es una escena del video: ejecútala y obtendrás un gráfico **interactivo** (zoom, pan, hover, selección).

**Índice**
1. Instalación y configuración
2. Scatter con hover y herramientas
3. Color por valor (colormap + colorbar)
4. Barras categóricas
5. Histograma y áreas
6. Leyenda interactiva
7. Series de tiempo con hover de fechas
8. Heatmap con `image`
9. Linked brushing (selección enlazada)
10. Widgets con `CustomJS` (slider)
11. Layouts: tabs y dashboard
12. Exportar a HTML

## 1️⃣ Instalación y configuración

In [1]:
!pip install -q --upgrade bokeh
import bokeh
print("Bokeh", bokeh.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
panel 1.9.4 requires bokeh<3.10.0,>=3.7.0, but you have bokeh 3.10.2 which is incompatible.
Bokeh 3.10.2


In [2]:
import numpy as np
import pandas as pd

from bokeh.io import output_notebook, show, save
from bokeh.plotting import figure
from bokeh.models import (ColumnDataSource, HoverTool, ColorBar, Slider, CustomJS,
                          TabPanel, Tabs, Div, BasicTicker)
from bokeh.layouts import row, column, gridplot
from bokeh.transform import linear_cmap, factor_cmap
from bokeh.palettes import Spectral6, Viridis256, Category10

output_notebook()   # <- renderiza los gráficos dentro de Colab
rng = np.random.default_rng(42)

## 2️⃣ Scatter con hover y herramientas
Pan, zoom, selección y *tooltips* sin escribir una línea de JavaScript.

In [3]:
n = 300
source = ColumnDataSource(dict(
    x=rng.normal(size=n),
    y=rng.normal(size=n),
    tam=rng.uniform(6, 22, n),
    grupo=rng.choice(["A", "B", "C"], n),
))

p = figure(width=700, height=420, title="Scatter interactivo",
           tools="pan,wheel_zoom,box_zoom,box_select,reset,save",
           tooltips=[("grupo", "@grupo"), ("x", "@x{0.00}"), ("y", "@y{0.00}")])

p.scatter("x", "y", size="tam", source=source, alpha=0.6,
          color=factor_cmap("grupo", palette=Category10[3], factors=["A", "B", "C"]))
p.xaxis.axis_label, p.yaxis.axis_label = "Variable X", "Variable Y"
show(p)

## 3️⃣ Color por valor (colormap + colorbar)

In [4]:
x = rng.uniform(0, 10, 400)
y = rng.uniform(0, 10, 400)
z = np.sin(x) * np.cos(y)

src = ColumnDataSource(dict(x=x, y=y, z=z))
mapper = linear_cmap("z", Viridis256, low=z.min(), high=z.max())

p = figure(width=700, height=450, title="Color según valor",
           tooltips=[("z", "@z{0.00}")], tools="pan,wheel_zoom,reset")
p.scatter("x", "y", source=src, size=10, color=mapper, line_color=None)
p.add_layout(ColorBar(color_mapper=mapper["transform"], width=12), "right")
show(p)

## 4️⃣ Barras categóricas

In [5]:
frutas = ["Manzana", "Pera", "Uva", "Mango", "Fresa", "Piña"]
ventas = [52, 31, 78, 64, 45, 90]

p = figure(x_range=frutas, width=700, height=400, title="Ventas por fruta",
           tooltips=[("fruta", "@fruta"), ("ventas", "@ventas")], toolbar_location=None)
p.vbar(x="fruta", top="ventas", width=0.8,
       source=ColumnDataSource(dict(fruta=frutas, ventas=ventas)),
       fill_color=factor_cmap("fruta", palette=Spectral6, factors=frutas), line_color="white")
p.y_range.start = 0
p.xgrid.grid_line_color = None
show(p)

## 5️⃣ Histograma y áreas

In [6]:
datos = rng.normal(loc=50, scale=12, size=2000)
hist, edges = np.histogram(datos, bins=40)

p1 = figure(width=380, height=350, title="Histograma", toolbar_location=None)
p1.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:],
        fill_color="#4c78a8", line_color="white", alpha=0.85)

xs = np.linspace(0, 10, 200)
p2 = figure(width=380, height=350, title="Área entre curvas", toolbar_location=None)
p2.varea(x=xs, y1=np.sin(xs), y2=np.sin(xs) + 1.5, fill_alpha=0.4, fill_color="#f58518")
p2.line(xs, np.sin(xs), line_width=2, color="#f58518")

show(row(p1, p2))

## 6️⃣ Leyenda interactiva
Haz clic en la leyenda para ocultar o atenuar series.

In [7]:
t = np.linspace(0, 4 * np.pi, 300)
p = figure(width=700, height=400, title="Haz clic en la leyenda")
for f, col in zip([np.sin, np.cos, np.tanh], Category10[3]):
    p.line(t, f(t), line_width=3, color=col, legend_label=f.__name__, muted_alpha=0.1)
p.legend.location = "top_left"
p.legend.click_policy = "mute"   # prueba también "hide"
show(p)

## 7️⃣ Series de tiempo con hover de fechas

In [8]:
fechas = pd.date_range("2025-01-01", periods=365, freq="D")
precio = 100 + np.cumsum(rng.normal(0, 1.5, 365))

p = figure(width=800, height=400, x_axis_type="datetime", title="Serie de tiempo",
           tools="pan,wheel_zoom,box_zoom,reset,save", x_range=(fechas[0], fechas[-1]))
p.line(fechas, precio, line_width=2, color="#2ca02c")
p.add_tools(HoverTool(tooltips=[("fecha", "@x{%F}"), ("valor", "@y{0.00}")],
                      formatters={"@x": "datetime"}, mode="vline"))
show(p)

## 8️⃣ Heatmap con `image`

In [9]:
N = 200
xx, yy = np.meshgrid(np.linspace(-3, 3, N), np.linspace(-3, 3, N))
Z = np.sin(xx**2 + yy**2) * np.exp(-0.1 * (xx**2 + yy**2))

p = figure(width=520, height=480, title="Heatmap", x_range=(-3, 3), y_range=(-3, 3))
p.image(image=[Z], x=-3, y=-3, dw=6, dh=6, palette="Turbo256")
show(p)

## 9️⃣ Linked brushing (selección enlazada)
Selecciona puntos en un gráfico y se resaltan en el otro. Comparten el mismo `ColumnDataSource`.

In [10]:
n = 250
a = rng.normal(size=n)
src = ColumnDataSource(dict(a=a, b=a * 0.6 + rng.normal(scale=0.6, size=n),
                            c=rng.normal(size=n)))

tools = "box_select,lasso_select,reset,wheel_zoom,pan"
p1 = figure(width=380, height=380, tools=tools, title="A vs B")
p2 = figure(width=380, height=380, tools=tools, title="A vs C")
p1.scatter("a", "b", source=src, size=8, alpha=0.7)
p2.scatter("a", "c", source=src, size=8, alpha=0.7)
show(gridplot([[p1, p2]]))

## 🔟 Widgets con `CustomJS`
El slider se ejecuta en el navegador (funciona directo en Colab, sin servidor).

In [11]:
x = np.linspace(0, 10, 500)
src = ColumnDataSource(dict(x=x, y=np.sin(x)))

p = figure(width=700, height=350, title="Onda interactiva", y_range=(-1.3, 1.3))
p.line("x", "y", source=src, line_width=3, color="#e45756")

slider = Slider(start=0.5, end=6, value=1, step=0.1, title="Frecuencia")
slider.js_on_change("value", CustomJS(args=dict(source=src, slider=slider), code="""
    const f = slider.value;
    const x = source.data.x;
    source.data = {x: x, y: Array.from(x, xi => Math.sin(f * xi))};
"""))
show(column(slider, p))

## 1️⃣1️⃣ Layouts: tabs y dashboard

In [12]:
def mini(titulo, color):
    g = figure(width=300, height=260, title=titulo, toolbar_location=None)
    xs = np.linspace(0, 10, 100)
    g.line(xs, np.sin(xs + rng.uniform(0, 3)), line_width=3, color=color)
    return g

g1, g2, g3 = mini("Ventas", "#4c78a8"), mini("Usuarios", "#f58518"), mini("Ingresos", "#54a24b")
encabezado = Div(text="<h2 style='margin:0'>📊 Mi dashboard con Bokeh</h2>")

tabs = Tabs(tabs=[TabPanel(child=row(g1, g2), title="Resumen"),
                  TabPanel(child=g3, title="Detalle")])
dashboard = column(encabezado, tabs)
show(dashboard)

## 1️⃣2️⃣ Exportar a HTML
Tu gráfico interactivo se convierte en una página que puedes compartir.

In [13]:
save(dashboard, filename="dashboard_bokeh.html", title="Dashboard Bokeh")

try:
    from google.colab import files
    files.download("dashboard_bokeh.html")
except ImportError:
    print("Guardado como dashboard_bokeh.html")

/tmp/ipykernel_1286/2842212554.py:1: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  save(dashboard, filename="dashboard_bokeh.html", title="Dashboard Bokeh")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### 💡 Extras para seguir
- `bokeh serve` permite callbacks en Python y *streaming* en vivo (requiere ejecutarlo fuera de Colab).
- `hvplot` y `Panel` se construyen sobre Bokeh.
- Documentación: https://docs.bokeh.org

**¡Sígueme en TikTok: Laboratorio de Códigos!** 🧪